<a href="https://colab.research.google.com/github/Yugansh-Varshney/ANN-Classification-Churn-Prediction-Model/blob/main/DL_Lab1_Data_to_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DL Lab 1 — From Cleaned Data to a Single ANN Forward Pass

**Goal of today:** you already know *what* an ANN is on paper. Today we make it real:
1. Load a real messy dataset and clean it (recap + practice)
2. See exactly how a cleaned row of data becomes an "input vector"
3. Build ONE forward pass through a tiny ANN by hand using NumPy — no PyTorch, no training yet
4. Understand `output = activation(W·x + b)` with real numbers

We are **not training** anything today. No backpropagation, no loss minimization. That's next lab.
Today is about making the architecture diagram feel like real numbers flowing through a real network.


## Step 1 — Setup (10–15 min)
Run this in Google Colab (colab.research.google.com) — no installs needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("numpy:", np.__version__)
print("pandas:", pd.__version__)


## Step 2 — Load the Dataset (5 min)
We'll use the **Titanic dataset** — it's small, famous, and (importantly) *messy* — perfect for practicing cleaning.
Each row = one passenger. Target: did they survive (1) or not (0)?


In [ ]:
import seaborn as sns

df = sns.load_dataset('titanic')
df.head()


### Instructor talking points (3 min)
Ask the class: *"What do you see wrong with this data already?"*
Let them look before you say anything. Expected answers: missing values, text columns (sex, embarked),
mixed types, columns that seem useless (deck, embark_town duplicate of embarked).


In [ ]:
df.info()


In [ ]:
df.isnull().sum().sort_values(ascending=False)


## Step 3 — Data Cleaning (35–40 min, hands-on)
This is the part you already taught in theory. Now they do it themselves, column by column.

**Have students do each of the following one at a time — pause after each and ask what changed.**


### 3.1 — Drop columns that are not useful as ANN inputs (redundant or too many missing)

In [ ]:
# 'deck' has too many missing values, 'embark_town'/'alive'/'class' duplicate other columns,
# 'who'/'adult_male' duplicate 'sex'/'age' info. Keep it simple for a first lab.
df_clean = df.drop(columns=['deck', 'embark_town', 'alive', 'class', 'who', 'adult_male', 'alone'])
df_clean.head()


### 3.2 — Handle missing values

In [ ]:
# 'age' has missing values -> fill with median (robust to outliers)
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())

# 'embarked' has 2 missing values -> fill with the most common port (mode)
df_clean['embarked'] = df_clean['embarked'].fillna(df_clean['embarked'].mode()[0])

df_clean.isnull().sum()


### 3.3 — Encode categorical (text) columns as numbers
ANNs only understand numbers. `sex` and `embarked` are text -> must be converted.


In [ ]:
# sex: binary -> simple 0/1 mapping
df_clean['sex'] = df_clean['sex'].map({'male': 0, 'female': 1})

# embarked: 3 categories (C, Q, S) -> one-hot encoding (no fake ordering implied)
df_clean = pd.get_dummies(df_clean, columns=['embarked'], drop_first=False)

df_clean.head()


### 3.4 — Scale numeric features
ANNs train better when input numbers are on a similar scale. We'll standardize manually (no sklearn) so the math is visible.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Select the columns to scale
columns_to_scale = ['age', 'fare']

# Apply standardization to the selected columns
df_clean[columns_to_scale] = scaler.fit_transform(df_clean[columns_to_scale])

df_clean[columns_to_scale].describe()

### 3.5 — Final check
By now every column should be numeric with no missing values — ready to feed into an ANN.


In [ ]:
df_clean.info()
df_clean.head()


## Step 4 — Bridging Data to ANN Architecture (10–15 min)
This is the key conceptual moment of the lab.

- Each **row** of `df_clean` (minus the target column `survived`) = **one input vector `x`**
- Each **feature/column** = **one input neuron**
- The number of remaining columns = **size of the input layer**


In [ ]:
X = df_clean.drop(columns=['survived']).values.astype(float)
y = df_clean['survived'].values.astype(float)

print("Shape of X (samples, features):", X.shape)
print("So the input layer of our ANN needs", X.shape[1], "neurons.")
print("\nFirst input vector x (one passenger):\n", X[0])


## Step 5 — Manual Forward Pass with NumPy (30–35 min, hands-on)
We'll build ONE hidden layer + ONE output neuron completely by hand.
No training yet — just: given random weights, what comes out the other end?


### 5.1 — Forward pass for a single sample

In [ ]:
from scipy.special import expit as sigmoid

n_features = X.shape[1]
n_hidden = 4   # you choose this it's a design decision, not derived from data

np.random.seed(42)
W1 = np.random.randn(n_hidden, n_features) * 0.1   # hidden layer weights
b1 = np.zeros(n_hidden)                             # hidden layer bias

x = X[0]                       # take the first passenger as our input vector
z1 = W1 @ x + b1               # linear step: W . x + b
a1 = sigmoid(z1)               # activation step

print("z1 (before activation):", z1)
print("a1 (after activation): ", a1)

- `W1 @ x` is literally what "fully connected layer" means: every input neuron connects to every hidden neuron.
- `W1` shape is `(n_hidden, n_features)` — each row of W1 is the weights feeding into ONE hidden neuron.
- `b1` shifts the result before squashing.
- `sigmoid` squashes any number into (0, 1) — this is the "activation."
- Right now the weights are **random** — the network knows nothing yet. That's expected. Training (next lab) is the process of adjusting W and b so the output becomes meaningful.


### 5.2 — Add the output layer (1 neuron, since this is binary classification: survived or not)

In [ ]:
n_output = 1

W2 = np.random.randn(n_output, n_hidden) * 0.1
b2 = np.zeros(n_output)

z2 = W2 @ a1 + b2
a2 = sigmoid(z2)

print("Final output (predicted probability of survival):", a2)
print("Actual label for this passenger:", y[0])


### 5.3 — Vectorize: do this for ALL passengers at once (not just one)
This is how it's actually done in practice — one matrix multiply instead of a loop.

In [ ]:
Z1 = X @ W1.T + b1        # shape: (n_samples, n_hidden)
A1 = sigmoid(Z1)

Z2 = A1 @ W2.T + b2       # shape: (n_samples, 1)
A2 = sigmoid(Z2)

print("A2 shape:", A2.shape)
print("First 5 predicted probabilities:\n", A2[:5].flatten())
print("First 5 actual labels:\n", y[:5])


## Step 6 — Wrap-up & Recap (10–15 min)

Walk through this chain out loud with the class, pointing at the code above for each step:

`raw messy data -> cleaned numeric data -> one row = input vector -> W1,b1 -> hidden layer -> activation -> W2,b2 -> output neuron -> activation -> prediction`

**Ask the class (discussion, no need to answer today):**
- Why were the predictions basically random / wrong?
- What do you think needs to change so predictions get better?
- What do W1 and W2 represent, physically?

**Next lab preview:** we'll introduce a *loss function* (how wrong are we?) and *gradient descent*
(how do we nudge W and b to be less wrong?) — using PyTorch instead of manual NumPy.

**Optional take-home exercise:** change `n_hidden` from 4 to 8 or 2, rerun Step 5, and see the output
shape doesn't change (why?) but the numbers do (why?).
